# EgoBrain — Visual Baselines & Temporal Robustness

**Purpose:** freeze the pure computer-vision reference on the locked subject-independent split before any neural supervision is considered.

This notebook trains the pooling, temporal-convolution and temporal-Transformer baselines, evaluates the six predefined temporal corruption families, records compute characteristics, and writes the canonical `sequence_index_with_split.csv` and model artifacts used downstream.

## 0. Experimental contract

The following are deliberately fixed before looking at model results:

| Item | Locked choice |
|---|---|
| Primary input | 512-D visual embedding per 1-second interval |
| Temporal context | `T = 8` consecutive seconds |
| Training sequence stride | 4 seconds |
| Evaluation sequence stride | 1 second |
| Frozen Phase-1 retained labels | 29 (27 semantic + 2 lifecycle) |
| Primary supervised action classes | 27 semantic action classes |
| Excluded from primary classifier | `unlabeled`, `ambiguous`, plus lifecycle-only `finish_running` and `last_stimuli` |
| Primary split | subject-grouped, stratified by class |
| Primary split construction | 5-fold StratifiedGroupKFold: fold 0 test, fold 1 validation, folds 2–4 train |
| Clean model selection | validation macro-F1 only; test set is never used for selection |
| Canonical dense model | Temporal Transformer (predeclared, not selected from test performance) |
| Robustness baseline | same Transformer trained with temporal masking augmentation |
| Corruption severities | 5/15/30/50/70% |
| Main reporting | accuracy, balanced accuracy, macro-F1, per-class metrics, confusion matrix, parameters, estimated FLOPs, inference latency, temporal length |
| Reproducibility | fixed random seed `20260927` |

### Label policy
The frozen Phase-1 retained vocabulary contains 27 semantic task classes plus `finish_running` and `last_stimuli`. The 29-label taxonomy remains authoritative, but `finish_running` and `last_stimuli` are lifecycle markers rather than sustained actions. Because they occur only ~once per subject and cannot yield valid 8-second same-label temporal windows, the primary Phase-2 action-recognition classifier uses the 27 semantic task classes. The lifecycle labels are audited and reported separately, never padded or synthetically expanded into training examples.

### Temporal-window policy
A training/evaluation example is formed only when all `T=8` rows are:
- from one subject,
- from one source video (`video_source_order`),
- temporally consecutive on the synchronized 1-second grid,
- and carry the same supervised label.

This avoids mixing actions or crossing video boundaries inside a supervised example.


In [ ]:
# ============================================================
# CELL 1 — CONFIGURATION / REPRODUCIBILITY LOCK
# ============================================================
from pathlib import Path
import os, json, math, time, hashlib, random, shutil, zipfile, warnings
from collections import OrderedDict

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.model_selection import StratifiedGroupKFold, GroupShuffleSplit
from sklearn.metrics import (
    accuracy_score, balanced_accuracy_score, f1_score,
    precision_recall_fscore_support, confusion_matrix,
)
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

# -----------------------------
# Global reproducibility
# -----------------------------
SEED = 20260927
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

# -----------------------------
# User's frozen Phase-1 input
# -----------------------------
PHASE1_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-egobrain-actual-phase-1-output"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase-1-output/egobrain_phase1"),
    Path("/kaggle/input/datasets/keerthikumarkj/egobrain-phase-1-output"),
]

VISUAL_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/keerthikumarkj/iris-egobrain-visual-features/uniform"),
]

OUT_ROOT = Path("/kaggle/working/egobrain_phase2")
MODELS_ROOT = OUT_ROOT / "models"
REPORTS_ROOT = OUT_ROOT / "reports"
FIGURES_ROOT = OUT_ROOT / "figures"
CORRUPTIONS_ROOT = OUT_ROOT / "corruptions"
for p in [OUT_ROOT, MODELS_ROOT, REPORTS_ROOT, FIGURES_ROOT, CORRUPTIONS_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

# -----------------------------
# Locked experimental constants
# -----------------------------
SUBJECTS = [f"P{i:04d}" for i in range(1, 41)]
VISUAL_DIM = 512
SEQ_LEN = 8
TRAIN_STRIDE = 4
EVAL_STRIDE = 1
BATCH_SIZE = 256
NUM_WORKERS = 0
MAX_EPOCHS = 15
PATIENCE = 4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0
STANDARDIZE_INPUT = True
CLASS_WEIGHT_POWER = 0.5

CORRUPTION_SEVERITIES = [0.05, 0.15, 0.30, 0.50, 0.70]
CORRUPTION_NAMES = [
    "random_frame_drop",
    "burst_deletion",
    "temporal_jitter",
    "irregular_sampling",
    "playback_speed_change",
    "contiguous_blackout",
]

# Canonical Phase-1 order: 27 semantic classes + 2 lifecycle classes.
SEMANTIC_TASK_CLASSES = [
    "draw_computer_pictures", "draw_pictures", "draw_tracing",
    "drink_bitter_gourd_juice", "drink_coca_cola", "drink_water",
    "eat_snack", "eat_spicy_strips", "eat_sugers",
    "operate_computer_excel", "operate_computer_powerpoint", "operate_computer_word",
    "play_cards", "play_computer_game_keyboard", "play_computer_game_mouse",
    "play_cube", "play_phone_game_complex", "play_phone_game_simple",
    "play_puzzle", "play_toy", "read_comic_book", "read_research_arcticle",
    "read_textbook", "watch_monitor", "watch_smartphone", "write_notes", "write_tracing",
]
LIFECYCLE_CLASSES = ["finish_running", "last_stimuli"]
PHASE1_RETAINED_CLASSES = SEMANTIC_TASK_CLASSES + LIFECYCLE_CLASSES
# Primary Phase-2 classifier: semantic actions only.
CLASS_NAMES = SEMANTIC_TASK_CLASSES
CLASS_TO_ID = {c: i for i, c in enumerate(CLASS_NAMES)}
ID_TO_CLASS = {i: c for c, i in CLASS_TO_ID.items()}
NUM_CLASSES = len(CLASS_NAMES)
PHASE1_RETAINED_CLASS_COUNT = len(PHASE1_RETAINED_CLASSES)
EXCLUDED_LABELS = {"unlabeled", "ambiguous"}
MODEL_EXCLUDED_LABELS = EXCLUDED_LABELS | set(LIFECYCLE_CLASSES)

# Canonical baseline names.
MODEL_NAMES = [
    "pool_mean",
    "pool_max",
    "pool_mean_max",
    "temporal_conv",
    "temporal_transformer",
    "temporal_mask_transformer",
]
CANONICAL_DENSE_MODEL = "temporal_transformer"
ROBUST_DENSE_MODEL = "temporal_mask_transformer"

# Training-model hyperparameters are fixed here, before training.
MODEL_CFG = {
    "pool_mean": {"lr": 2e-3},
    "pool_max": {"lr": 2e-3},
    "pool_mean_max": {"lr": 2e-3},
    "temporal_conv": {"lr": 1e-3, "channels": 192},
    "temporal_transformer": {"lr": 3e-4, "d_model": 256, "heads": 8, "layers": 4, "ff_dim": 512, "dropout": 0.10},
    "temporal_mask_transformer": {"lr": 3e-4, "d_model": 256, "heads": 8, "layers": 4, "ff_dim": 512, "dropout": 0.10, "train_mask_rate": 0.20},
}

RUN_CROSS_SCENE = True
RUN_WITHIN_SUBJECT_DIAGNOSTIC = False  # optional supporting diagnostic; primary paper result remains cross-subject.
RUN_CORRUPTION_ALL_MODELS = True
RESUME_EXISTING_MODELS = True

print("Locked temporal length:", SEQ_LEN)
print("Frozen Phase-1 retained labels:", PHASE1_RETAINED_CLASS_COUNT)
print("Locked primary supervised action classes:", NUM_CLASSES)
print("Models:", MODEL_NAMES)
print("Corruption severities:", CORRUPTION_SEVERITIES)


In [ ]:
# ============================================================
# CELL 2 — RESOLVE FROZEN INPUTS (NO PHASE-1 RECOMPUTATION)
# ============================================================

def first_existing(paths):
    for p in paths:
        if p.exists():
            return p
    return None

PHASE1_ROOT = first_existing(PHASE1_ROOT_CANDIDATES)
VISUAL_ROOT = first_existing(VISUAL_ROOT_CANDIDATES)

if PHASE1_ROOT is None:
    raise FileNotFoundError(
        "Could not find the uploaded Phase-1 artifact dataset. Checked:\n" +
        "\n".join(str(p) for p in PHASE1_ROOT_CANDIDATES)
    )
if VISUAL_ROOT is None:
    raise FileNotFoundError(
        "Could not find the existing 512-D visual feature dataset. Checked:\n" +
        "\n".join(str(p) for p in VISUAL_ROOT_CANDIDATES)
    )


def find_phase1_file(filename):
    direct = PHASE1_ROOT / filename
    if direct.exists():
        return direct
    hits = sorted(PHASE1_ROOT.rglob(filename))
    if hits:
        return hits[0]
    return None

# Prefer the labeled hand-off. The plain synchronized index is still
# validated and retained as provenance, but modeling requires action_label.
LABELED_INDEX = find_phase1_file("all_subjects_synchronized_labeled_index.csv")
PLAIN_INDEX = find_phase1_file("all_subjects_synchronized_index.csv")
VIDEO_METADATA = find_phase1_file("video_metadata.csv")
PHASE1_MANIFEST = find_phase1_file("phase1_manifest.json")

print("Phase-1 root:", PHASE1_ROOT)
print("Visual root:", VISUAL_ROOT)
print("Labeled index:", LABELED_INDEX)
print("Plain index:", PLAIN_INDEX)
print("Video metadata:", VIDEO_METADATA)
print("Phase-1 manifest:", PHASE1_MANIFEST)

if LABELED_INDEX is None:
    raise FileNotFoundError(
        "Phase 2 requires all_subjects_synchronized_labeled_index.csv. "
        "The unlabeled synchronized index alone cannot provide supervised targets."
    )

if PLAIN_INDEX is None:
    print("WARNING: plain synchronized index was not found; labeled index remains the modeling hand-off.")

phase1_index = pd.read_csv(LABELED_INDEX)
required_cols = {
    "subject_id", "sync_index", "visual_feature_idx", "start_sec", "end_sec", "action_label",
}
missing = required_cols - set(phase1_index.columns)
if missing:
    raise RuntimeError(f"Frozen Phase-1 hand-off is missing required columns: {sorted(missing)}")

print("Frozen hand-off rows:", len(phase1_index))
print("Subjects:", phase1_index.subject_id.nunique())
print("Columns:", list(phase1_index.columns))


In [ ]:
# ============================================================
# CELL 3 — FROZEN-HANDOFF VALIDATION + LABEL DISTRIBUTION
# ============================================================

assert phase1_index.subject_id.nunique() == 40, "Expected all 40 subjects in the frozen hand-off."
assert set(phase1_index.subject_id.unique()) == set(SUBJECTS), "Subject coverage is not exactly P0001-P0040."
assert not phase1_index.duplicated(["subject_id", "sync_index"]).any(), "Duplicate synchronized indices detected."

# Never silently transform labels.
phase1_index["action_label"] = phase1_index["action_label"].astype(str)

observed_labels = set(phase1_index.action_label.unique())
allowed_phase1_labels = set(PHASE1_RETAINED_CLASSES) | EXCLUDED_LABELS
unknown_labels = observed_labels - allowed_phase1_labels
if unknown_labels:
    raise RuntimeError(f"Unexpected action labels in frozen index: {sorted(unknown_labels)}")

print("Observed label states:")
print(phase1_index.action_label.value_counts(dropna=False).to_string())

# Preserve the full Phase-1 29-label audit, but exclude lifecycle markers from
# the Phase-2 temporal action-recognition classifier. They are isolated markers
# and cannot yield valid 8-second same-label action windows.
lifecycle_rows = phase1_index[phase1_index.action_label.isin(LIFECYCLE_CLASSES)].copy()
lifecycle_rows.action_label.value_counts().rename_axis("action_label").to_frame("rows").to_csv(
    REPORTS_ROOT / "phase1_lifecycle_label_audit.csv"
)

valid = phase1_index[~phase1_index.action_label.isin(MODEL_EXCLUDED_LABELS)].copy()
missing_classes = set(CLASS_NAMES) - set(valid.action_label.unique())
if missing_classes:
    raise RuntimeError(f"Primary 27-class semantic action task is missing labels: {sorted(missing_classes)}")

# Temporal integrity needed for sequence formation.
valid = valid.sort_values(["subject_id", "sync_index"]).reset_index(drop=True)

if "video_source_order" not in valid.columns:
    raise RuntimeError(
        "Phase-1 hand-off does not contain video_source_order; Phase 2 refuses to create "
        "temporal examples without a video-boundary key."
    )

valid["video_source_order"] = pd.to_numeric(valid["video_source_order"], errors="raise")
valid["sync_index"] = pd.to_numeric(valid["sync_index"], errors="raise")
valid["visual_feature_idx"] = pd.to_numeric(valid["visual_feature_idx"], errors="raise")
valid["start_sec"] = pd.to_numeric(valid["start_sec"], errors="raise")
valid["end_sec"] = pd.to_numeric(valid["end_sec"], errors="raise")

# Retain provenance columns whenever available.
for col in ["video_remote_path", "video_clip_idx", "label_conflict", "stage1_count_method"]:
    if col not in valid.columns:
        valid[col] = np.nan

print("Rows entering primary semantic-action classifier:", len(valid))
print("Rows excluded from primary action classifier:", len(phase1_index) - len(valid))
print("  unlabeled/ambiguous:", int(phase1_index.action_label.isin(EXCLUDED_LABELS).sum()))
print("  lifecycle markers:", int(phase1_index.action_label.isin(LIFECYCLE_CLASSES).sum()))
print("Primary semantic action class count:", valid.action_label.nunique())
print("Frozen Phase-1 retained label count:", PHASE1_RETAINED_CLASS_COUNT)

class_counts = valid.action_label.value_counts().reindex(CLASS_NAMES)
class_counts.to_csv(REPORTS_ROOT / "supervised_class_distribution.csv", header=["rows"])

fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(np.arange(NUM_CLASSES), class_counts.values)
ax.set_xticks(np.arange(NUM_CLASSES))
ax.set_xticklabels(CLASS_NAMES, rotation=75, ha="right", fontsize=8)
ax.set_ylabel("Synchronized 1-second rows")
ax.set_title("Phase-2 supervised label distribution before sequence construction")
fig.tight_layout()
fig.savefig(FIGURES_ROOT / "01_supervised_class_distribution.png", dpi=160)
plt.show()


### Critical Phase-2 label decision — lifecycle markers are audit-only

`finish_running` (39 rows) and `last_stimuli` (40 rows) are present in the frozen Phase-1 taxonomy, but they are not sustained actions. With the locked `T=8` seconds and the rule that every timestep in a supervised sequence must share one label, these markers cannot produce scientifically valid training sequences.

**Therefore:** visual-baseline stage preserves all 29 Phase-1 labels in the audit, but the actual visual action-recognition model is a **27-class semantic-action classifier**. No padding, duplication, or synthetic lifecycle windows are introduced.


In [ ]:
# ============================================================
# CELL 4 — LOAD VISUAL TENSORS + VALIDATE ALL 40 SUBJECTS
# ============================================================


def find_subject_visual_file(subject):
    exact = VISUAL_ROOT / f"features_{subject}.pt"
    if exact.exists():
        return exact
    candidates = sorted(VISUAL_ROOT.glob(f"*{subject}*.pt"))
    if not candidates:
        raise FileNotFoundError(f"No 512-D visual tensor found for {subject}")
    return candidates[0]


def load_tensor_object(path):
    obj = torch.load(path, map_location="cpu", weights_only=False)
    if isinstance(obj, dict) and "features" in obj:
        return obj["features"], obj
    return obj, {}

visual_cache = {}
visual_inventory = []

for subject in SUBJECTS:
    path = find_subject_visual_file(subject)
    tensor, meta = load_tensor_object(path)
    if not isinstance(tensor, torch.Tensor):
        raise TypeError(f"{subject}: visual object is not a torch.Tensor")
    if tensor.ndim != 2 or tensor.shape[1] != VISUAL_DIM:
        raise RuntimeError(f"{subject}: expected [N,512], got {tuple(tensor.shape)}")
    if not torch.isfinite(tensor.float()).all():
        raise RuntimeError(f"{subject}: visual tensor contains NaN/Inf")
    visual_cache[subject] = tensor.cpu()
    visual_inventory.append({
        "subject_id": subject,
        "path": str(path),
        "rows": int(tensor.shape[0]),
        "dim": int(tensor.shape[1]),
        "dtype": str(tensor.dtype),
    })

visual_inventory = pd.DataFrame(visual_inventory)
visual_inventory.to_csv(REPORTS_ROOT / "visual_tensor_inventory.csv", index=False)

display(visual_inventory)
print("Validated visual tensors:", len(visual_cache), "/ 40")
print("Total visual feature rows:", int(visual_inventory.rows.sum()))


In [ ]:
# ============================================================
# CELL 5 — BUILD FIXED 8-SECOND SUPERVISED SEQUENCES
# ============================================================


def contiguous_segments(group, key_cols=("sync_index", "visual_feature_idx")):
    g = group.sort_values("sync_index").reset_index(drop=False)
    if len(g) == 0:
        return []
    sync = g["sync_index"].to_numpy()
    vis = g["visual_feature_idx"].to_numpy()
    # A segment is continuous only if both the synchronized grid and the
    # underlying visual feature index advance by one row.
    good = (np.diff(sync) == 1) & (np.diff(vis) == 1)
    breaks = np.where(~good)[0] + 1
    return np.split(g, breaks)


def build_sequence_table(df, stride):
    records = []
    segment_count = 0
    dropped_short_segments = 0
    for (subject, video_order, label), g in df.groupby(
        ["subject_id", "video_source_order", "action_label"], sort=False
    ):
        for seg in contiguous_segments(g):
            segment_count += 1
            n = len(seg)
            if n < SEQ_LEN:
                dropped_short_segments += 1
                continue
            for offset in range(0, n - SEQ_LEN + 1, stride):
                s = seg.iloc[offset: offset + SEQ_LEN]
                starts = s["start_sec"].to_numpy(dtype=np.float64)
                ends = s["end_sec"].to_numpy(dtype=np.float64)
                if not np.allclose(np.diff(starts), 1.0, atol=1e-6):
                    continue
                if not np.allclose(ends - starts, 1.0, atol=1e-6):
                    continue
                records.append({
                    "subject_id": subject,
                    "video_source_order": int(video_order),
                    "action_label": label,
                    "label_id": CLASS_TO_ID[label],
                    "sync_start": int(s.iloc[0]["sync_index"]),
                    "sync_end": int(s.iloc[-1]["sync_index"]),
                    "visual_start": int(s.iloc[0]["visual_feature_idx"]),
                    "visual_end": int(s.iloc[-1]["visual_feature_idx"]),
                    "start_sec": float(s.iloc[0]["start_sec"]),
                    "end_sec": float(s.iloc[-1]["end_sec"]),
                    "video_remote_path": s.iloc[0].get("video_remote_path", np.nan),
                })
    out = pd.DataFrame(records)
    return out, segment_count, dropped_short_segments

sequence_tables = {}
for split_stride_name, stride in [("train_candidate", TRAIN_STRIDE), ("eval_candidate", EVAL_STRIDE)]:
    seq, nseg, ndrop = build_sequence_table(valid, stride)
    sequence_tables[split_stride_name] = seq
    print(split_stride_name, "rows:", len(seq), "segments:", nseg, "short segments dropped:", ndrop)

# The evaluation table uses stride 1. A later split is performed once on this
# master table, then the training subset is downsampled to the locked train stride.
all_seq_eval = sequence_tables["eval_candidate"].copy()

# Train-stride selection is deterministic and happens before the subject split.
train_stride_keys = set(
    zip(
        sequence_tables["train_candidate"]["subject_id"],
        sequence_tables["train_candidate"]["video_source_order"],
        sequence_tables["train_candidate"]["sync_start"],
        sequence_tables["train_candidate"]["action_label"],
    )
)

# Keep a single canonical evaluation sequence index; split assignment is done below.
all_seq_eval.to_csv(OUT_ROOT / "sequence_index_eval_stride1.csv", index=False)
sequence_tables["train_candidate"].to_csv(OUT_ROOT / "sequence_index_train_stride4.csv", index=False)

def split_subject_sequences(seq_df, seed=SEED):
    if len(seq_df) == 0:
        raise RuntimeError("No supervised temporal sequences were constructed.")
    splitter = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    X_dummy = np.zeros((len(seq_df), 1), dtype=np.float32)
    y = seq_df["label_id"].to_numpy()
    groups = seq_df["subject_id"].to_numpy()
    folds = list(splitter.split(X_dummy, y, groups))
    # Fold 0 = test, fold 1 = validation, folds 2-4 = train.
    train_idx = np.concatenate([folds[i][1] for i in [2, 3, 4]])
    val_idx = folds[1][1]
    test_idx = folds[0][1]
    assign = np.full(len(seq_df), "", dtype=object)
    assign[train_idx] = "train"
    assign[val_idx] = "val"
    assign[test_idx] = "test"
    assert np.all(assign != "")
    out = seq_df.copy()
    out["split"] = assign
    return out

split_seq = split_subject_sequences(all_seq_eval)

train_subjects = sorted(split_seq.loc[split_seq.split == "train", "subject_id"].unique())
val_subjects = sorted(split_seq.loc[split_seq.split == "val", "subject_id"].unique())
test_subjects = sorted(split_seq.loc[split_seq.split == "test", "subject_id"].unique())

if set(train_subjects) & set(val_subjects) or set(train_subjects) & set(test_subjects) or set(val_subjects) & set(test_subjects):
    raise RuntimeError("Subject leakage detected in primary split.")

# Construct the actual train table by selecting only starts that match the fixed stride-4 table.
train_key = pd.MultiIndex.from_frame(
    sequence_tables["train_candidate"][["subject_id", "video_source_order", "sync_start", "action_label"]]
)
key_eval = pd.MultiIndex.from_frame(
    split_seq[["subject_id", "video_source_order", "sync_start", "action_label"]]
)
train_membership = key_eval.isin(train_key)
split_seq.loc[(split_seq.split == "train") & (~train_membership), "split"] = "train_stride_excluded"

train_seq = split_seq[split_seq.split == "train"].copy()
val_seq = split_seq[split_seq.split == "val"].copy()
test_seq = split_seq[split_seq.split == "test"].copy()

split_seq.to_csv(OUT_ROOT / "sequence_index_with_split.csv", index=False)

split_manifest = pd.DataFrame([
    {"split": "train", "subjects": ",".join(train_subjects), "num_subjects": len(train_subjects), "num_sequences": len(train_seq)},
    {"split": "val", "subjects": ",".join(val_subjects), "num_subjects": len(val_subjects), "num_sequences": len(val_seq)},
    {"split": "test", "subjects": ",".join(test_subjects), "num_subjects": len(test_subjects), "num_sequences": len(test_seq)},
])
split_manifest.to_csv(REPORTS_ROOT / "primary_subject_split.csv", index=False)

print("Primary subject split")
print("TRAIN:", train_subjects)
print("VAL  :", val_subjects)
print("TEST :", test_subjects)
print()
print("Sequence counts:", {"train": len(train_seq), "val": len(val_seq), "test": len(test_seq)})
print("Train-stride exclusions from stride-1 master:", int((split_seq.split == "train_stride_excluded").sum()))


In [ ]:
# ============================================================
# CELL 6 — TRAIN-ONLY FEATURE STANDARDIZATION + CLASS WEIGHTS
# ============================================================

# Feature normalization uses only visual tensors belonging to TRAIN subjects.
# No labels from validation/test are touched while computing statistics.

if STANDARDIZE_INPUT:
    sum_vec = torch.zeros(VISUAL_DIM, dtype=torch.float64)
    sum_sq_vec = torch.zeros(VISUAL_DIM, dtype=torch.float64)
    count = 0
    for subject in train_subjects:
        x = visual_cache[subject].float()
        sum_vec += x.sum(dim=0, dtype=torch.float64)
        sum_sq_vec += (x.double() ** 2).sum(dim=0)
        count += x.shape[0]
    mean = sum_vec / max(count, 1)
    var = (sum_sq_vec / max(count, 1)) - (mean ** 2)
    std = torch.sqrt(torch.clamp(var, min=1e-8)).float()
    mean = mean.float()
else:
    mean = torch.zeros(VISUAL_DIM)
    std = torch.ones(VISUAL_DIM)

np.save(OUT_ROOT / "train_feature_mean.npy", mean.numpy())
np.save(OUT_ROOT / "train_feature_std.npy", std.numpy())

train_counts = train_seq["label_id"].value_counts().reindex(range(NUM_CLASSES), fill_value=0).sort_index()
if (train_counts == 0).any():
    missing_train = [ID_TO_CLASS[i] for i in train_counts[train_counts == 0].index]
    raise RuntimeError(f"Primary training split lacks classes: {missing_train}")

# Pandas Series does not expose Tensor.float(); convert the locked
# training counts to a torch tensor before applying the class-weight rule.
train_count_tensor = torch.tensor(train_counts.to_numpy(dtype=np.float32), dtype=torch.float32)
weights = (train_count_tensor.max() / train_count_tensor).pow(CLASS_WEIGHT_POWER)
weights = weights / weights.mean()
class_weights = weights.to(torch.float32)
torch.save(class_weights, OUT_ROOT / "class_weights.pt")
print("Class-weight construction: torch tensor from training counts; no validation/test statistics used.")

print("Train-stat standardization:", STANDARDIZE_INPUT)
print("Train rows used for stats:", count if STANDARDIZE_INPUT else "n/a")
print("Train sequence counts per class:")
print(pd.Series({ID_TO_CLASS[i]: int(train_counts[i]) for i in range(NUM_CLASSES)}).to_string())


In [ ]:
# ============================================================
# CELL 6B — SANITY CHECK CLASS WEIGHTS
# ============================================================
assert isinstance(class_weights, torch.Tensor)
assert class_weights.shape == (NUM_CLASSES,)
assert torch.isfinite(class_weights).all().item()
assert (class_weights > 0).all().item()
print("PASS: class_weights tensor:", tuple(class_weights.shape), "dtype=", class_weights.dtype)


In [ ]:
# ============================================================
# CELL 7 — DATASET / DATALOADERS
# ============================================================

class VisualSequenceDataset(Dataset):
    def __init__(self, sequence_df, visual_cache, mean, std, seq_len=SEQ_LEN):
        self.df = sequence_df.reset_index(drop=True).copy()
        self.visual_cache = visual_cache
        self.mean = mean
        self.std = std
        self.seq_len = seq_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        subject = row.subject_id
        start = int(row.visual_start)
        x = self.visual_cache[subject][start:start + self.seq_len].float()
        if x.shape != (self.seq_len, VISUAL_DIM):
            raise RuntimeError(f"Bad visual slice at {subject}:{start}: {tuple(x.shape)}")
        x = (x - self.mean) / self.std
        y = int(row.label_id)
        return x, y, idx

train_ds = VisualSequenceDataset(train_seq, visual_cache, mean, std)
val_ds = VisualSequenceDataset(val_seq, visual_cache, mean, std)
test_ds = VisualSequenceDataset(test_seq, visual_cache, mean, std)

loader_kwargs = dict(batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type == "cuda"))
train_loader = DataLoader(train_ds, shuffle=True, drop_last=False, **loader_kwargs)
val_loader = DataLoader(val_ds, shuffle=False, drop_last=False, **loader_kwargs)
test_loader = DataLoader(test_ds, shuffle=False, drop_last=False, **loader_kwargs)

print("Dataloaders:", len(train_loader), len(val_loader), len(test_loader), "batches")


In [ ]:
# ============================================================
# CELL 8 — MODEL DEFINITIONS
# ============================================================

class PoolingClassifier(nn.Module):
    def __init__(self, mode="mean", dim=VISUAL_DIM, num_classes=NUM_CLASSES):
        super().__init__()
        assert mode in {"mean", "max", "mean_max"}
        self.mode = mode
        in_dim = dim if mode != "mean_max" else 2 * dim
        self.classifier = nn.Linear(in_dim, num_classes)

    def forward(self, x):
        # x: [B,T,D]
        if self.mode == "mean":
            z = x.mean(dim=1)
        elif self.mode == "max":
            z = x.max(dim=1).values
        else:
            z = torch.cat([x.mean(dim=1), x.max(dim=1).values], dim=-1)
        return self.classifier(z)


class TemporalConvClassifier(nn.Module):
    def __init__(self, dim=VISUAL_DIM, channels=192, num_classes=NUM_CLASSES):
        super().__init__()
        self.input = nn.Conv1d(dim, channels, kernel_size=1)
        self.block1 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=1),
            nn.BatchNorm1d(channels), nn.GELU(), nn.Dropout(0.10),
        )
        self.block2 = nn.Sequential(
            nn.Conv1d(channels, channels, kernel_size=3, padding=2, dilation=2),
            nn.BatchNorm1d(channels), nn.GELU(), nn.Dropout(0.10),
        )
        self.classifier = nn.Linear(channels, num_classes)

    def forward(self, x):
        z = x.transpose(1, 2)
        z = self.input(z)
        z = z + self.block1(z)
        z = z + self.block2(z)
        z = z.mean(dim=-1)
        return self.classifier(z)


class SinusoidalPositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=SEQ_LEN):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
        div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div)
        pe[:, 1::2] = torch.cos(position * div)
        self.register_buffer("pe", pe.unsqueeze(0), persistent=False)

    def forward(self, x):
        return x + self.pe[:, :x.shape[1]]


class TemporalTransformerClassifier(nn.Module):
    def __init__(self, dim=VISUAL_DIM, d_model=256, heads=8, layers=4, ff_dim=512, dropout=0.10, num_classes=NUM_CLASSES):
        super().__init__()
        self.input = nn.Linear(dim, d_model)
        self.pos = SinusoidalPositionalEncoding(d_model, max_len=SEQ_LEN)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=heads,
            dim_feedforward=ff_dim,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=layers)
        self.norm = nn.LayerNorm(d_model)
        self.classifier = nn.Linear(d_model, num_classes)

    def forward(self, x, train_mask_rate=0.0):
        z = self.input(x)
        if self.training and train_mask_rate > 0:
            # Conventional temporal masking: zero selected tokens after projection.
            keep = (torch.rand(z.shape[:2], device=z.device) >= train_mask_rate).float().unsqueeze(-1)
            z = z * keep
        z = self.pos(z)
        z = self.encoder(z)
        z = self.norm(z.mean(dim=1))
        return self.classifier(z)


def build_model(name):
    if name == "pool_mean":
        return PoolingClassifier("mean")
    if name == "pool_max":
        return PoolingClassifier("max")
    if name == "pool_mean_max":
        return PoolingClassifier("mean_max")
    if name == "temporal_conv":
        return TemporalConvClassifier(channels=MODEL_CFG[name]["channels"])
    if name == "temporal_transformer":
        return TemporalTransformerClassifier(
            d_model=MODEL_CFG[name]["d_model"],
            heads=MODEL_CFG[name]["heads"],
            layers=MODEL_CFG[name]["layers"],
            ff_dim=MODEL_CFG[name]["ff_dim"],
            dropout=MODEL_CFG[name]["dropout"],
        )
    if name == "temporal_mask_transformer":
        return TemporalTransformerClassifier(
            d_model=MODEL_CFG[name]["d_model"],
            heads=MODEL_CFG[name]["heads"],
            layers=MODEL_CFG[name]["layers"],
            ff_dim=MODEL_CFG[name]["ff_dim"],
            dropout=MODEL_CFG[name]["dropout"],
        )
    raise KeyError(name)

# Smoke-test every architecture before training.
for name in MODEL_NAMES:
    model = build_model(name).to(DEVICE)
    x = torch.randn(2, SEQ_LEN, VISUAL_DIM, device=DEVICE)
    with torch.no_grad():
        y = model(x, train_mask_rate=(MODEL_CFG[name].get("train_mask_rate", 0.0) if "transformer" in name else 0.0))
    assert y.shape == (2, NUM_CLASSES), (name, y.shape)
    print(f"PASS {name}: {tuple(y.shape)}")
    del model, x, y
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


### Baseline definitions

**A — pooling:** no temporal mixing beyond a fixed summary statistic. Mean/max/mean+max are intentionally simple reference points.

**B — temporal convolution:** local temporal interactions with two residual 1-D convolutional blocks.

**C — temporal Transformer:** the canonical dense temporal model. This is the predeclared visual backbone against which later adaptive routing and EEG augmentation will be compared.

**D — temporal-mask Transformer:** the same Transformer capacity, but trained with randomly masked temporal tokens. This is a conventional corruption-robust training baseline; it is **not** claimed as a novelty.


In [ ]:
# ============================================================
# CELL 9 — METRICS / FLOPs / LATENCY HELPERS
# ============================================================

def metrics_from_predictions(y_true, y_pred):
    acc = accuracy_score(y_true, y_pred)
    bacc = balanced_accuracy_score(y_true, y_pred)
    macro = f1_score(y_true, y_pred, average="macro", zero_division=0)
    weighted = f1_score(y_true, y_pred, average="weighted", zero_division=0)
    return {
        "accuracy": float(acc),
        "balanced_accuracy": float(bacc),
        "macro_f1": float(macro),
        "weighted_f1": float(weighted),
    }


def per_class_frame(y_true, y_pred):
    p, r, f, s = precision_recall_fscore_support(
        y_true, y_pred, labels=np.arange(NUM_CLASSES), zero_division=0
    )
    return pd.DataFrame({
        "class_id": np.arange(NUM_CLASSES),
        "class_name": [ID_TO_CLASS[i] for i in range(NUM_CLASSES)],
        "precision": p,
        "recall": r,
        "f1": f,
        "support": s,
        "class_type": ["semantic_action"] * NUM_CLASSES,
    })


def count_params(model):
    return int(sum(p.numel() for p in model.parameters()))


def estimate_forward_flops(model_name):
    T, D, C = SEQ_LEN, VISUAL_DIM, NUM_CLASSES
    if model_name == "pool_mean":
        return int(2 * D * C + T * D)
    if model_name == "pool_max":
        return int(2 * D * C + T * D)
    if model_name == "pool_mean_max":
        return int(4 * D * C + 2 * T * D)
    if model_name == "temporal_conv":
        ch = MODEL_CFG[model_name]["channels"]
        # Conv MACs * 2, plus pointwise/BN ignored for simplicity.
        conv1 = 2 * T * D * ch
        conv2 = 2 * T * ch * ch * 3
        conv3 = 2 * T * ch * ch * 3
        fc = 2 * ch * C
        return int(conv1 + conv2 + conv3 + fc)
    if model_name in {"temporal_transformer", "temporal_mask_transformer"}:
        d = MODEL_CFG[model_name]["d_model"]
        h = MODEL_CFG[model_name]["heads"]
        L = MODEL_CFG[model_name]["layers"]
        ff = MODEL_CFG[model_name]["ff_dim"]
        input_proj = 2 * T * D * d
        per_layer = (
            2 * T * d * 3 * d +      # qkv
            4 * T * T * d +          # qk^T + attention*v
            2 * T * d * d +          # output projection
            2 * (2 * T * d * ff)      # FFN two linears
        )
        cls = 2 * d * C
        return int(input_proj + L * per_layer + cls)
    return -1


def benchmark_latency(model, device=DEVICE, batches=20):
    model.eval()
    x = torch.randn(BATCH_SIZE, SEQ_LEN, VISUAL_DIM, device=device)
    # Warm-up.
    with torch.no_grad():
        for _ in range(5):
            _ = model(x)
    if device.type == "cuda":
        torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.no_grad():
            for _ in range(batches):
                _ = model(x)
        torch.cuda.synchronize()
    else:
        start = time.perf_counter()
        with torch.no_grad():
            for _ in range(batches):
                _ = model(x)
    elapsed = time.perf_counter() - start
    ms_batch = elapsed * 1000.0 / batches
    ms_sample = ms_batch / BATCH_SIZE
    del x
    return float(ms_sample)


def save_confusion(y_true, y_pred, model_name, split_name):
    cm = confusion_matrix(y_true, y_pred, labels=np.arange(NUM_CLASSES))
    cm_df = pd.DataFrame(cm, index=CLASS_NAMES, columns=CLASS_NAMES)
    cm_dir = REPORTS_ROOT / model_name
    cm_dir.mkdir(parents=True, exist_ok=True)
    cm_df.to_csv(cm_dir / f"confusion_matrix_{split_name}.csv")

    fig, ax = plt.subplots(figsize=(12, 10))
    im = ax.imshow(cm, aspect="auto")
    fig.colorbar(im, ax=ax)
    ax.set_xticks(np.arange(NUM_CLASSES))
    ax.set_yticks(np.arange(NUM_CLASSES))
    ax.set_xticklabels(CLASS_NAMES, rotation=90, fontsize=6)
    ax.set_yticklabels(CLASS_NAMES, fontsize=6)
    ax.set_xlabel("Predicted")
    ax.set_ylabel("True")
    ax.set_title(f"{model_name} — {split_name} confusion matrix")
    fig.tight_layout()
    fig.savefig(cm_dir / f"confusion_matrix_{split_name}.png", dpi=160)
    plt.close(fig)


In [ ]:
# ============================================================
# CELL 10 — TRAIN / EVAL ENGINE
# ============================================================

from contextlib import nullcontext


def autocast_context():
    if DEVICE.type != "cuda":
        return nullcontext()
    # torch.amp.autocast is available on modern Kaggle PyTorch;
    # keep a fallback for older images.
    try:
        return torch.autocast(device_type="cuda", dtype=torch.float16)
    except Exception:
        return torch.cuda.amp.autocast()


def train_one_model(model_name):
    model_dir = MODELS_ROOT / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    ckpt_path = model_dir / "best_checkpoint.pt"
    history_path = model_dir / "history.csv"

    model = build_model(model_name).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    optimizer = torch.optim.AdamW(
        model.parameters(), lr=MODEL_CFG[model_name]["lr"], weight_decay=WEIGHT_DECAY
    )
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=MAX_EPOCHS)

    # Resume only from our own checkpoint with the same model name.
    start_epoch = 0
    best_val = -np.inf
    best_state = None
    history = []
    if RESUME_EXISTING_MODELS and ckpt_path.exists():
        try:
            payload = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
            model.load_state_dict(payload["model_state"])
            optimizer.load_state_dict(payload["optimizer_state"])
            if "scheduler_state" in payload:
                scheduler.load_state_dict(payload["scheduler_state"])
            start_epoch = int(payload.get("epoch", 0)) + 1
            best_val = float(payload.get("best_val_macro_f1", -np.inf))
            history = payload.get("history", [])
            if payload.get("best_model_state") is not None:
                best_state = {k: v.detach().cpu().clone() for k, v in payload["best_model_state"].items()}
            print(f"RESUME {model_name}: starting at epoch {start_epoch}, best val macro-F1={best_val:.5f}")
        except Exception as exc:
            print("Checkpoint resume skipped:", exc)

    if start_epoch >= MAX_EPOCHS and best_state is not None:
        model.load_state_dict(best_state)
        torch.save(
            {"model_name": model_name, "model_state": model.state_dict(), "config": MODEL_CFG[model_name], "seed": SEED},
            model_dir / "best_weights_only.pt"
        )
        return model

    scaler = None
    if DEVICE.type == "cuda":
        try:
            scaler = torch.amp.GradScaler("cuda")
        except Exception:
            scaler = torch.cuda.amp.GradScaler()

    patience_count = 0
    for epoch in range(start_epoch, MAX_EPOCHS):
        model.train()
        epoch_loss = 0.0
        seen = 0
        for xb, yb, _ in train_loader:
            xb = xb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)
            optimizer.zero_grad(set_to_none=True)
            mask_rate = MODEL_CFG[model_name].get("train_mask_rate", 0.0) if model_name == ROBUST_DENSE_MODEL else 0.0
            with autocast_context():
                logits = model(xb, train_mask_rate=mask_rate) if "transformer" in model_name else model(xb)
                loss = criterion(logits, yb)
            if scaler is not None:
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
                optimizer.step()
            epoch_loss += float(loss.detach().cpu()) * len(yb)
            seen += len(yb)

        scheduler.step()

        val_result = evaluate_model(model, val_loader, model_name, split_name="val", save_outputs=False)
        train_loss = epoch_loss / max(seen, 1)
        row = {
            "epoch": epoch,
            "train_loss": train_loss,
            "val_accuracy": val_result["metrics"]["accuracy"],
            "val_balanced_accuracy": val_result["metrics"]["balanced_accuracy"],
            "val_macro_f1": val_result["metrics"]["macro_f1"],
            "lr": optimizer.param_groups[0]["lr"],
        }
        history.append(row)
        pd.DataFrame(history).to_csv(history_path, index=False)

        current = row["val_macro_f1"]
        print(
            f"{model_name:26s} | epoch {epoch+1:02d}/{MAX_EPOCHS} | "
            f"loss={train_loss:.4f} | val_macroF1={current:.4f} | "
            f"val_bacc={row['val_balanced_accuracy']:.4f}"
        )

        if current > best_val + 1e-6:
            best_val = current
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1

        payload = {
            "model_name": model_name,
            "epoch": epoch,
            "best_val_macro_f1": best_val,
            "model_state": model.state_dict(),
            "best_model_state": best_state,
            "optimizer_state": optimizer.state_dict(),
            "scheduler_state": scheduler.state_dict(),
            "history": history,
            "config": MODEL_CFG[model_name],
            "seed": SEED,
        }
        torch.save(payload, ckpt_path)

        if patience_count >= PATIENCE:
            print(f"EARLY STOP {model_name}: patience reached.")
            break

    # Load the best state seen in this run/resume.
    if best_state is not None:
        model.load_state_dict(best_state)
    elif ckpt_path.exists():
        payload = torch.load(ckpt_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(payload["model_state"])

    torch.save(
        {"model_name": model_name, "model_state": model.state_dict(), "config": MODEL_CFG[model_name], "seed": SEED},
        model_dir / "best_weights_only.pt"
    )
    return model


def evaluate_model(model, loader, model_name, split_name, save_outputs=True):
    model.eval()
    y_true, y_pred = [], []
    all_idx = []
    with torch.no_grad():
        for xb, yb, idx in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            with autocast_context():
                logits = model(xb)
            pred = logits.argmax(dim=1).cpu().numpy()
            y_pred.extend(pred.tolist())
            y_true.extend(yb.numpy().tolist())
            all_idx.extend(idx.numpy().tolist())

    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    metrics = metrics_from_predictions(y_true, y_pred)
    per_class = per_class_frame(y_true, y_pred)

    if save_outputs:
        out_dir = REPORTS_ROOT / model_name
        out_dir.mkdir(parents=True, exist_ok=True)
        with open(out_dir / f"{split_name}_metrics.json", "w") as f:
            json.dump(metrics, f, indent=2)
        per_class.to_csv(out_dir / f"{split_name}_per_class.csv", index=False)
        save_confusion(y_true, y_pred, model_name, split_name)

    return {
        "metrics": metrics,
        "y_true": y_true,
        "y_pred": y_pred,
        "indices": np.asarray(all_idx, dtype=np.int64),
        "per_class": per_class,
    }


## 3. Train the complete clean-data baseline ladder

The same primary subject split, preprocessing statistics, temporal length, target vocabulary, loss weighting, and evaluation code are used for every model. Only the temporal architecture/training strategy changes.

The three pooling baselines deliberately provide a low-complexity floor; the TCN measures local temporal modeling; the dense Transformer establishes the main visual ceiling; and the mask-trained Transformer tests a conventional robustness intervention before any adaptive routing is introduced.


In [ ]:
# ============================================================
# CELL 11 — TRAIN ALL CLEAN VISUAL BASELINES
# ============================================================

trained_models = OrderedDict()
model_records = []

for model_name in MODEL_NAMES:
    print("\n" + "=" * 90)
    print("TRAINING", model_name)
    print("=" * 90)
    model = train_one_model(model_name)
    trained_models[model_name] = model

    val = evaluate_model(model, val_loader, model_name, "val", save_outputs=True)
    test = evaluate_model(model, test_loader, model_name, "test", save_outputs=True)
    latency = benchmark_latency(model)
    params = count_params(model)
    flops = estimate_forward_flops(model_name)

    record = {
        "model": model_name,
        "parameters": params,
        "estimated_forward_flops_per_sample": flops,
        "inference_latency_ms_per_sample": latency,
        "temporal_length_seconds": SEQ_LEN,
        "val_accuracy": val["metrics"]["accuracy"],
        "val_balanced_accuracy": val["metrics"]["balanced_accuracy"],
        "val_macro_f1": val["metrics"]["macro_f1"],
        "test_accuracy": test["metrics"]["accuracy"],
        "test_balanced_accuracy": test["metrics"]["balanced_accuracy"],
        "test_macro_f1": test["metrics"]["macro_f1"],
        "test_weighted_f1": test["metrics"]["weighted_f1"],
    }
    model_records.append(record)
    pd.DataFrame(model_records).to_csv(REPORTS_ROOT / "clean_model_summary.csv", index=False)

clean_summary = pd.DataFrame(model_records).sort_values("test_macro_f1", ascending=False)
print("\nCLEAN TEST SUMMARY")
display(clean_summary)


In [ ]:
# ============================================================
# CELL 12 — LOCK THE CANONICAL DENSE VISUAL BASELINE
# ============================================================

# This is intentionally NOT chosen by test performance.
# The temporal Transformer was predeclared in the experimental contract.
canonical_name = CANONICAL_DENSE_MODEL
robust_name = ROBUST_DENSE_MODEL

canonical_metrics_path = REPORTS_ROOT / canonical_name / "test_metrics.json"
if not canonical_metrics_path.exists():
    raise RuntimeError("Canonical Transformer test result is missing.")

with open(canonical_metrics_path) as f:
    canonical_test_metrics = json.load(f)

canonical_val = clean_summary.loc[clean_summary.model == canonical_name].iloc[0]

problem_lock = {
    "phase": "phase_2",
    "status": "LOCKED",
    "task": "Egocentric action recognition from temporally ordered visual features",
    "input_dim": VISUAL_DIM,
    "temporal_length_seconds": SEQ_LEN,
    "frozen_phase1_retained_label_count": PHASE1_RETAINED_CLASS_COUNT,
    "frozen_phase1_retained_labels": PHASE1_RETAINED_CLASSES,
    "primary_supervised_action_class_count": NUM_CLASSES,
    "classes": CLASS_NAMES,
    "excluded_from_primary_classifier": sorted(MODEL_EXCLUDED_LABELS),
    "train_stride_seconds": TRAIN_STRIDE,
    "evaluation_stride_seconds": EVAL_STRIDE,
    "primary_split": "StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=20260927): fold0=test, fold1=val, folds2-4=train",
    "canonical_dense_model": canonical_name,
    "robustness_baseline": robust_name,
    "clean_selection_metric": "validation_macro_f1",
    "test_metrics_are_never_used_for_model_selection": True,
    "lifecycle_labels_excluded_from_primary_classifier": LIFECYCLE_CLASSES,
    "eeg_used": False,
    "corruption_severities": CORRUPTION_SEVERITIES,
}
with open(OUT_ROOT / "problem_lock.json", "w") as f:
    json.dump(problem_lock, f, indent=2)

print("PROBLEM LOCKED")
print("Canonical dense visual model:", canonical_name)
print("Canonical test metrics:", canonical_test_metrics)


In [ ]:
# ============================================================
# CELL 13 — CORRUPTION FUNCTIONS (FIXED BEFORE RESULTS)
# ============================================================

# All corruption operators accept normalized [B,T,D] features and preserve
# the final tensor shape. They are deterministic conditional on torch's RNG
# state. No corruption parameter is tuned from model results.


def _random_keep_mask(B, T, rate, device):
    keep = torch.rand(B, T, device=device) >= rate
    return keep


def corrupt_random_frame_drop(x, rate):
    B, T, D = x.shape
    keep = _random_keep_mask(B, T, rate, x.device)
    out = torch.zeros_like(x)
    for b in range(B):
        kept = x[b, keep[b]]
        n = min(T, kept.shape[0])
        out[b, :n] = kept[:n]
    return out


def corrupt_burst_deletion(x, rate):
    B, T, D = x.shape
    burst = max(1, int(math.ceil(rate * T)))
    out = torch.zeros_like(x)
    for b in range(B):
        if burst >= T:
            continue
        start = int(torch.randint(0, T - burst + 1, (1,), device=x.device).item())
        kept = torch.cat([x[b, :start], x[b, start + burst:]], dim=0)
        n = min(T, kept.shape[0])
        out[b, :n] = kept[:n]
    return out


def corrupt_blackout(x, rate):
    B, T, D = x.shape
    block = max(1, int(math.ceil(rate * T)))
    out = x.clone()
    for b in range(B):
        if block >= T:
            out[b] = 0
            continue
        start = int(torch.randint(0, T - block + 1, (1,), device=x.device).item())
        out[b, start:start + block] = 0
    return out


def corrupt_temporal_jitter(x, rate):
    B, T, D = x.shape
    radius = max(1, int(round(rate * T / 2)))
    base = torch.arange(T, device=x.device).view(1, T).expand(B, T)
    offsets = torch.randint(-radius, radius + 1, (B, T), device=x.device)
    idx = torch.clamp(base + offsets, 0, T - 1)
    return torch.gather(x, 1, idx.unsqueeze(-1).expand(-1, -1, D))


def _resample_sequence(x, positions):
    # Linear interpolation along the temporal axis.
    B, T, D = x.shape
    pos = positions.clamp(0, T - 1)
    lo = pos.floor().long()
    hi = pos.ceil().long()
    alpha = (pos - lo.float()).unsqueeze(-1)
    lo_x = torch.gather(x, 1, lo.unsqueeze(-1).expand(-1, -1, D))
    hi_x = torch.gather(x, 1, hi.unsqueeze(-1).expand(-1, -1, D))
    return lo_x * (1.0 - alpha) + hi_x * alpha


def corrupt_irregular_sampling(x, rate):
    B, T, D = x.shape
    base = torch.linspace(0, T - 1, T, device=x.device).view(1, T).expand(B, T)
    jitter = torch.randn(B, T, device=x.device) * (rate * 0.75)
    pos = torch.sort(torch.clamp(base + jitter, 0, T - 1), dim=1).values
    return _resample_sequence(x, pos)


def corrupt_playback_speed(x, rate):
    B, T, D = x.shape
    # Random speed multiplier in [1-rate, 1+rate].
    speed = 1.0 + (2.0 * torch.rand(B, 1, device=x.device) - 1.0) * rate
    center = (T - 1) / 2.0
    base = torch.arange(T, device=x.device).view(1, T).expand(B, T)
    positions = center + (base - center) * speed
    return _resample_sequence(x, positions)

CORRUPTION_FUNCS = {
    "random_frame_drop": corrupt_random_frame_drop,
    "burst_deletion": corrupt_burst_deletion,
    "temporal_jitter": corrupt_temporal_jitter,
    "irregular_sampling": corrupt_irregular_sampling,
    "playback_speed_change": corrupt_playback_speed,
    "contiguous_blackout": corrupt_blackout,
}

# Smoke test shape preservation.
probe = torch.randn(4, SEQ_LEN, VISUAL_DIM, device=DEVICE)
for cname, fn in CORRUPTION_FUNCS.items():
    for sev in [0.05, 0.70]:
        z = fn(probe, sev)
        assert z.shape == probe.shape, (cname, sev, z.shape)
    print("PASS", cname)
del probe


### Corruption semantics

The six perturbations are implemented as **evaluation attacks**, not novelty claims:

- **Random frame/clip drop:** selected temporal embeddings are removed and the remaining tokens are left-packed with zero padding.
- **Burst deletion:** a contiguous temporal burst is removed and the remainder is left-packed with zero padding.
- **Temporal jitter:** each sample is reassigned a nearby temporal position within a severity-dependent jitter radius.
- **Irregular sampling:** the temporal grid is warped into nonuniform sample positions with interpolation.
- **Playback-speed change:** the same sequence is resampled at a fixed random speed multiplier in `[1-s, 1+s]`.
- **Contiguous blackout:** a contiguous region is retained in place but replaced by the neutral zero embedding.

All five severity levels are declared before observing results. The corruption benchmark never changes the test labels.


In [ ]:
# ============================================================
# CELL 14 — CORRUPTION EVALUATION
# ============================================================

def evaluate_corruptions(model, model_name, loader, max_batches=None):
    rows = []
    model.eval()
    # Fixed seed at the start of each full sweep makes the stochastic corruption
    # realization reproducible across models for the same corruption/severity.
    for cname in CORRUPTION_NAMES:
        fn = CORRUPTION_FUNCS[cname]
        for severity in CORRUPTION_SEVERITIES:
            torch.manual_seed(SEED + int(severity * 1000) + CORRUPTION_NAMES.index(cname) * 10000)
            if torch.cuda.is_available():
                torch.cuda.manual_seed_all(SEED + int(severity * 1000) + CORRUPTION_NAMES.index(cname) * 10000)
            y_true, y_pred = [], []
            batch_count = 0
            with torch.no_grad():
                for xb, yb, _ in loader:
                    xb = xb.to(DEVICE, non_blocking=True)
                    xb = fn(xb, severity)
                    with autocast_context():
                        logits = model(xb)
                    y_true.extend(yb.numpy().tolist())
                    y_pred.extend(logits.argmax(dim=1).cpu().numpy().tolist())
                    batch_count += 1
                    if max_batches is not None and batch_count >= max_batches:
                        break
            m = metrics_from_predictions(np.asarray(y_true), np.asarray(y_pred))
            rows.append({
                "model": model_name,
                "corruption": cname,
                "severity": severity,
                **m,
            })
            print(f"{model_name:26s} | {cname:22s} | {severity:>4.0%} | macro-F1={m['macro_f1']:.4f} | bAcc={m['balanced_accuracy']:.4f}")
    return pd.DataFrame(rows)

corruption_frames = []
models_for_corruption = MODEL_NAMES if RUN_CORRUPTION_ALL_MODELS else [CANONICAL_DENSE_MODEL, ROBUST_DENSE_MODEL]
for model_name in models_for_corruption:
    print("\n" + "=" * 90)
    print("CORRUPTION EVALUATION:", model_name)
    print("=" * 90)
    df = evaluate_corruptions(trained_models[model_name], model_name, test_loader)
    corruption_frames.append(df)

corruption_results = pd.concat(corruption_frames, ignore_index=True)
corruption_results.to_csv(REPORTS_ROOT / "corruption_benchmark.csv", index=False)

deficiency = corruption_results.copy()
deficiency["macro_f1_drop_from_clean"] = np.nan
for model_name in deficiency.model.unique():
    clean = float(clean_summary.loc[clean_summary.model == model_name, "test_macro_f1"].iloc[0])
    idx = deficiency.model == model_name
    deficiency.loc[idx, "macro_f1_drop_from_clean"] = clean - deficiency.loc[idx, "macro_f1"]
deficiency.to_csv(REPORTS_ROOT / "corruption_benchmark_with_clean_drop.csv", index=False)


In [ ]:
# ============================================================
# CELL 15 — CORRUPTION CURVES / ROBUSTNESS REPORT
# ============================================================

summary_rows = []
for model_name in corruption_results.model.unique():
    sub = corruption_results[corruption_results.model == model_name]
    clean_macro = float(clean_summary.loc[clean_summary.model == model_name, "test_macro_f1"].iloc[0])
    clean_bacc = float(clean_summary.loc[clean_summary.model == model_name, "test_balanced_accuracy"].iloc[0])
    for cname in CORRUPTION_NAMES:
        c = sub[sub.corruption == cname].sort_values("severity")
        for _, row in c.iterrows():
            summary_rows.append({
                "model": model_name,
                "corruption": cname,
                "severity": row.severity,
                "macro_f1": row.macro_f1,
                "balanced_accuracy": row.balanced_accuracy,
                "macro_f1_drop": clean_macro - row.macro_f1,
                "balanced_accuracy_drop": clean_bacc - row.balanced_accuracy,
            })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(REPORTS_ROOT / "corruption_summary.csv", index=False)

# One plot per corruption family for readability.
for cname in CORRUPTION_NAMES:
    fig, ax = plt.subplots(figsize=(8, 5))
    for model_name in corruption_results.model.unique():
        c = summary_df[(summary_df.model == model_name) & (summary_df.corruption == cname)].sort_values("severity")
        ax.plot(c.severity * 100, c.macro_f1, marker="o", label=model_name)
    ax.set_xlabel("Severity (%)")
    ax.set_ylabel("Macro-F1")
    ax.set_title(f"Temporal corruption — {cname}")
    ax.grid(True, alpha=0.25)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(FIGURES_ROOT / f"corruption_{cname}.png", dpi=160)
    plt.close(fig)

display(summary_df.head(30))


In [ ]:
# ============================================================
# CELL 16 — CROSS-SCENE SUPPORTING EVALUATION (ONLY IF METADATA SUPPORTS IT)
# ============================================================

cross_scene_status = {"status": "NOT_RUN", "reason": None}
cross_scene_results = []

SCENE_COLUMN_CANDIDATES = [
    "scene", "scene_id", "scene_name", "environment", "environment_id",
    "location", "location_id", "room", "room_id"
]

if not RUN_CROSS_SCENE:
    cross_scene_status["status"] = "DISABLED_BY_CONFIG"
    cross_scene_status["reason"] = "RUN_CROSS_SCENE=False"
elif VIDEO_METADATA is None:
    cross_scene_status["status"] = "NOT_AVAILABLE"
    cross_scene_status["reason"] = "Frozen Phase-1 video_metadata.csv was not included in the mounted artifact input."
else:
    video_meta = pd.read_csv(VIDEO_METADATA)
    scene_cols = [c for c in SCENE_COLUMN_CANDIDATES if c in video_meta.columns]
    if not scene_cols:
        cross_scene_status["status"] = "NOT_AVAILABLE"
        cross_scene_status["reason"] = "No explicit scene/environment/location field exists in the mounted Phase-1 video metadata; Phase 2 refuses to invent one."
    else:
        scene_col = scene_cols[0]
        if "video_source_order" not in video_meta.columns:
            cross_scene_status["status"] = "NOT_AVAILABLE"
            cross_scene_status["reason"] = "video_metadata.csv lacks video_source_order, so synchronized rows cannot be assigned to scenes safely."
        else:
            # Attach the explicit scene field through the frozen source-order mapping.
            join_cols = ["video_source_order", scene_col]
            if "subject_id" in video_meta.columns:
                join_cols.append("subject_id")
            meta_small = video_meta[join_cols].drop_duplicates()
            # Cross-scene is a separate environment diagnostic over the full
            # synchronized supervised sequence pool. It does not alter the
            # primary subject-held-out split or any primary model checkpoint.
            seq_scene = all_seq_eval.copy()
            # Use source order + subject when available.
            keys = ["video_source_order", "subject_id"] if "subject_id" in meta_small.columns else ["video_source_order"]
            seq_scene = seq_scene.merge(meta_small, on=keys, how="left", validate="many_to_one")
            if seq_scene[scene_col].isna().any():
                cross_scene_status["status"] = "NOT_AVAILABLE"
                cross_scene_status["reason"] = "Some supervised sequences could not be mapped unambiguously to the explicit scene field."
            elif seq_scene[scene_col].nunique() < 3:
                cross_scene_status["status"] = "NOT_AVAILABLE"
                cross_scene_status["reason"] = f"Only {seq_scene[scene_col].nunique()} scene groups exist; a meaningful held-out scene evaluation requires at least 3."
            else:
                # Scene-held-out diagnostic: fold 0 scenes = test, fold 1 scenes = val,
                # remaining scenes = train. This measures environment generalization;
                # subject leakage is intentionally documented because this is a scene,
                # not a subject, diagnostic.
                scenes = sorted(seq_scene[scene_col].astype(str).unique())
                splitter = GroupShuffleSplit(n_splits=1, test_size=0.20, random_state=SEED)
                all_rows = np.arange(len(seq_scene))
                train_tmp_idx, test_idx = next(splitter.split(all_rows, groups=seq_scene[scene_col].astype(str)))
                train_tmp = seq_scene.iloc[train_tmp_idx].copy()
                held_test = seq_scene.iloc[test_idx].copy()
                splitter2 = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=SEED + 1)
                tr2, va2 = next(splitter2.split(np.arange(len(train_tmp)), groups=train_tmp[scene_col].astype(str)))
                held_train = train_tmp.iloc[tr2].copy()
                held_val = train_tmp.iloc[va2].copy()

                # Use the same model family, optimizer, class weights and feature stats.
                # The scene diagnostic intentionally uses the canonical dense Transformer only.
                scene_model = build_model(CANONICAL_DENSE_MODEL).to(DEVICE)
                scene_train_ds = VisualSequenceDataset(held_train, visual_cache, mean, std)
                scene_val_ds = VisualSequenceDataset(held_val, visual_cache, mean, std)
                scene_test_ds = VisualSequenceDataset(held_test, visual_cache, mean, std)
                scene_train_loader = DataLoader(scene_train_ds, shuffle=True, **loader_kwargs)
                scene_val_loader = DataLoader(scene_val_ds, shuffle=False, **loader_kwargs)
                scene_test_loader = DataLoader(scene_test_ds, shuffle=False, **loader_kwargs)
                scene_opt = torch.optim.AdamW(scene_model.parameters(), lr=MODEL_CFG[CANONICAL_DENSE_MODEL]["lr"], weight_decay=WEIGHT_DECAY)
                scene_crit = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
                best_scene = -np.inf
                best_scene_state = None
                patience = 0
                for epoch in range(MAX_EPOCHS):
                    scene_model.train()
                    for xb, yb, _ in scene_train_loader:
                        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                        scene_opt.zero_grad(set_to_none=True)
                        with autocast_context():
                            loss = scene_crit(scene_model(xb), yb)
                        loss.backward()
                        nn.utils.clip_grad_norm_(scene_model.parameters(), GRAD_CLIP)
                        scene_opt.step()
                    v = evaluate_model(scene_model, scene_val_loader, "cross_scene_temp", "val", save_outputs=False)["metrics"]
                    if v["macro_f1"] > best_scene:
                        best_scene = v["macro_f1"]
                        best_scene_state = {k: t.detach().cpu().clone() for k, t in scene_model.state_dict().items()}
                        patience = 0
                    else:
                        patience += 1
                    print(f"cross-scene Transformer | epoch {epoch+1:02d} | val macro-F1={v['macro_f1']:.4f}")
                    if patience >= PATIENCE:
                        break
                scene_model.load_state_dict(best_scene_state)
                s = evaluate_model(scene_model, scene_test_loader, "cross_scene_temp", "test", save_outputs=False)["metrics"]
                cross_scene_results.append({
                    "model": CANONICAL_DENSE_MODEL,
                    "scene_column": scene_col,
                    "num_scenes_total": len(scenes),
                    "train_scene_count": held_train[scene_col].nunique(),
                    "val_scene_count": held_val[scene_col].nunique(),
                    "test_scene_count": held_test[scene_col].nunique(),
                    "train_subjects": held_train.subject_id.nunique(),
                    "val_subjects": held_val.subject_id.nunique(),
                    "test_subjects": held_test.subject_id.nunique(),
                    **s,
                })
                cross_scene_status["status"] = "EVALUATED"
                cross_scene_status["scene_column"] = scene_col
                cross_scene_status["train_scenes"] = sorted(held_train[scene_col].astype(str).unique())
                cross_scene_status["val_scenes"] = sorted(held_val[scene_col].astype(str).unique())
                cross_scene_status["test_scenes"] = sorted(held_test[scene_col].astype(str).unique())

with open(REPORTS_ROOT / "cross_scene_status.json", "w") as f:
    json.dump(cross_scene_status, f, indent=2)

if cross_scene_results:
    cross_scene_df = pd.DataFrame(cross_scene_results)
    cross_scene_df.to_csv(REPORTS_ROOT / "cross_scene_results.csv", index=False)
    display(cross_scene_df)
else:
    print("Cross-scene status:", cross_scene_status)


### Cross-scene interpretation rule

This notebook does **not** infer a scene from filename order, subject ID, session number, or arbitrary source ordering. Cross-scene evaluation runs only when the frozen Phase-1 video metadata contains an explicit scene/environment/location variable that can be joined to `video_source_order`.

If that information is absent, the correct scientific result is **“cross-scene not available from the frozen metadata”**, not a fabricated scene split.


In [ ]:
# ============================================================
# CELL 17 — OPTIONAL WITHIN-SUBJECT DIAGNOSTIC
# ============================================================

within_subject_status = {"status": "DISABLED_BY_CONFIG" if not RUN_WITHIN_SUBJECT_DIAGNOSTIC else "NOT_RUN"}

if RUN_WITHIN_SUBJECT_DIAGNOSTIC:
    # This is deliberately a diagnostic only. It is not a paper primary result.
    # Split each subject's sequence examples into 80/20 random train/test subsets,
    # then report the canonical Transformer. Subject identity appears in both sets.
    rng = np.random.default_rng(SEED)
    ws_train_parts, ws_test_parts = [], []
    for subject, g in all_seq_eval.groupby("subject_id"):
        g = g.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
        n = max(1, int(0.8 * len(g)))
        ws_train_parts.append(g.iloc[:n])
        ws_test_parts.append(g.iloc[n:])
    ws_train = pd.concat(ws_train_parts, ignore_index=True)
    ws_test = pd.concat(ws_test_parts, ignore_index=True)
    ws_train_ds = VisualSequenceDataset(ws_train, visual_cache, mean, std)
    ws_test_ds = VisualSequenceDataset(ws_test, visual_cache, mean, std)
    ws_train_loader = DataLoader(ws_train_ds, shuffle=True, **loader_kwargs)
    ws_test_loader = DataLoader(ws_test_ds, shuffle=False, **loader_kwargs)

    ws_model = build_model(CANONICAL_DENSE_MODEL).to(DEVICE)
    ws_opt = torch.optim.AdamW(ws_model.parameters(), lr=MODEL_CFG[CANONICAL_DENSE_MODEL]["lr"], weight_decay=WEIGHT_DECAY)
    ws_crit = nn.CrossEntropyLoss(weight=class_weights.to(DEVICE))
    for epoch in range(MAX_EPOCHS):
        ws_model.train()
        for xb, yb, _ in ws_train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            ws_opt.zero_grad(set_to_none=True)
            with autocast_context():
                loss = ws_crit(ws_model(xb), yb)
            loss.backward()
            nn.utils.clip_grad_norm_(ws_model.parameters(), GRAD_CLIP)
            ws_opt.step()
    ws_metrics = evaluate_model(ws_model, ws_test_loader, "within_subject_temp", "test", save_outputs=False)["metrics"]
    pd.DataFrame([ws_metrics]).to_csv(REPORTS_ROOT / "within_subject_diagnostic.csv", index=False)
    within_subject_status = {"status": "EVALUATED", **ws_metrics}

with open(REPORTS_ROOT / "within_subject_status.json", "w") as f:
    json.dump(within_subject_status, f, indent=2)
print("Within-subject diagnostic:", within_subject_status)


In [ ]:
# ============================================================
# CELL 18 — FINAL PHASE-2 GATE
# ============================================================

clean_summary = pd.read_csv(REPORTS_ROOT / "clean_model_summary.csv")
corruption_results = pd.read_csv(REPORTS_ROOT / "corruption_benchmark.csv")

checks = OrderedDict()
checks["all_40_subjects_present"] = len(set(train_subjects + val_subjects + test_subjects)) == 40
checks["no_subject_overlap"] = not (
    (set(train_subjects) & set(val_subjects)) or
    (set(train_subjects) & set(test_subjects)) or
    (set(val_subjects) & set(test_subjects))
)
checks["all_27_semantic_classes_in_train"] = set(train_seq.label_id.unique()) == set(range(NUM_CLASSES))
checks["phase1_lifecycle_audit_exists"] = (REPORTS_ROOT / "phase1_lifecycle_label_audit.csv").exists()
checks["all_models_completed"] = set(clean_summary.model) == set(MODEL_NAMES)
checks["canonical_transformer_reported"] = CANONICAL_DENSE_MODEL in set(clean_summary.model)
checks["robustness_transformer_reported"] = ROBUST_DENSE_MODEL in set(clean_summary.model)
checks["all_corruptions_present"] = set(corruption_results.corruption) == set(CORRUPTION_NAMES)
checks["all_severities_present"] = set(np.round(corruption_results.severity.astype(float), 4)) == set(np.round(CORRUPTION_SEVERITIES, 4))
checks["corruption_model_coverage"] = set(corruption_results.model) == set(MODEL_NAMES if RUN_CORRUPTION_ALL_MODELS else [CANONICAL_DENSE_MODEL, ROBUST_DENSE_MODEL])
checks["clean_metric_files_exist"] = all((REPORTS_ROOT / m / "test_metrics.json").exists() for m in MODEL_NAMES)
checks["problem_lock_exists"] = (OUT_ROOT / "problem_lock.json").exists()

# Cross-scene is a conditional supporting experiment, so it does not fail the gate
# when the frozen metadata genuinely lacks an explicit scene/environment field.
with open(REPORTS_ROOT / "cross_scene_status.json") as f:
    css = json.load(f)
checks["cross_scene_condition_handled"] = css.get("status") in {"EVALUATED", "NOT_AVAILABLE", "DISABLED_BY_CONFIG"}

# Strong canonical dense baseline must have non-null clean metrics.
canonical_row = clean_summary[clean_summary.model == CANONICAL_DENSE_MODEL].iloc[0]
checks["canonical_clean_metrics_finite"] = bool(np.isfinite(canonical_row.test_macro_f1) and np.isfinite(canonical_row.test_balanced_accuracy))

phase2_pass = all(checks.values())
status = "PASS" if phase2_pass else "INCOMPLETE"

phase2_gate = {
    "phase": "phase_2",
    "status": status,
    "goal_achieved": [
        "clean video-only baseline established",
        "progressive temporal baseline ladder evaluated",
        "primary cross-subject split locked and leakage-audited",
        "fixed temporal corruption benchmark evaluated",
        "canonical dense visual backbone locked before EEG",
    ],
    "checks": checks,
    "canonical_dense_model": CANONICAL_DENSE_MODEL,
    "canonical_dense_test_metrics": canonical_row.to_dict(),
    "cross_scene_status": css,
    "output_root": str(OUT_ROOT),
    "phase3_prerequisite": bool(phase2_pass),
}
with open(REPORTS_ROOT / "phase2_gate_report.json", "w") as f:
    json.dump(phase2_gate, f, indent=2, default=lambda x: bool(x) if isinstance(x, (np.bool_,)) else x)

# Human-readable report.
lines = []
lines.append("EgoBrain PHASE 2 — FINAL GATE")
lines.append("=" * 72)
lines.append(f"STATUS: {status}")
lines.append("")
lines.append("Primary problem: Egocentric action recognition from temporally ordered 512-D visual features")
lines.append(f"Temporal context: {SEQ_LEN} seconds")
lines.append(f"Primary split: subject-grouped | train={len(train_subjects)} subjects | val={len(val_subjects)} | test={len(test_subjects)}")
lines.append(f"Canonical dense visual model: {CANONICAL_DENSE_MODEL}")
lines.append(f"Primary semantic action classes: {NUM_CLASSES} | Frozen Phase-1 retained labels: {PHASE1_RETAINED_CLASS_COUNT}")
lines.append("Lifecycle markers finish_running/last_stimuli are audit-only because they cannot form valid T=8 same-label windows.")
lines.append("")
for k, v in checks.items():
    lines.append(f"{'PASS' if v else 'FAIL':4s}  {k}")
lines.append("")
lines.append("Canonical dense clean test metrics:")
for k in ["test_accuracy", "test_balanced_accuracy", "test_macro_f1", "test_weighted_f1", "parameters", "estimated_forward_flops_per_sample", "inference_latency_ms_per_sample"]:
    lines.append(f"  {k}: {canonical_row[k]}")
lines.append("")
lines.append(f"Cross-scene status: {css.get('status')} — {css.get('reason', '')}")
lines.append("")
if phase2_pass:
    lines.append("PHASE 2 CLEARED — READY TO MOVE TO PHASE 3 (EEG INTRODUCTION / MULTIMODAL EXPERIMENTS)")
else:
    lines.append("PHASE 2 NOT YET CLEARED — inspect failed checks before Phase 3")

with open(REPORTS_ROOT / "phase2_gate_report.txt", "w") as f:
    f.write("\n".join(lines))

print("\n".join(lines))


In [ ]:
# ============================================================
# CELL 19 — PACKAGE PHASE-2 ARTIFACTS
# ============================================================

# Keep raw input tensors outside the package. The package contains only
# reproducibility metadata, sequence indices, reports, figures and checkpoints.
PACKAGE_BASE = OUT_ROOT.parent / "egobrain_phase2_artifacts"
ZIP_PATH = OUT_ROOT.parent / "egobrain_phase2_artifacts.zip"

if PACKAGE_BASE.exists():
    shutil.rmtree(PACKAGE_BASE)
PACKAGE_BASE.mkdir(parents=True)

for src in [
    OUT_ROOT / "problem_lock.json",
    OUT_ROOT / "sequence_index_eval_stride1.csv",
    OUT_ROOT / "sequence_index_train_stride4.csv",
    OUT_ROOT / "sequence_index_with_split.csv",
    OUT_ROOT / "train_feature_mean.npy",
    OUT_ROOT / "train_feature_std.npy",
    OUT_ROOT / "class_weights.pt",
    REPORTS_ROOT,
    FIGURES_ROOT,
    MODELS_ROOT,
]:
    if src.exists():
        dest = PACKAGE_BASE / src.name
        if src.is_dir():
            shutil.copytree(src, dest)
        else:
            shutil.copy2(src, dest)

# Add a compact run manifest.
run_manifest = {
    "phase": "phase_2",
    "seed": SEED,
    "device": str(DEVICE),
    "subjects": SUBJECTS,
    "sequence_length": SEQ_LEN,
    "train_stride": TRAIN_STRIDE,
    "eval_stride": EVAL_STRIDE,
    "phase1_retained_class_names": PHASE1_RETAINED_CLASSES,
    "primary_supervised_action_class_names": CLASS_NAMES,
    "models": MODEL_NAMES,
    "canonical_model": CANONICAL_DENSE_MODEL,
    "robust_model": ROBUST_DENSE_MODEL,
    "corruptions": CORRUPTION_NAMES,
    "severities": CORRUPTION_SEVERITIES,
    "phase1_labeled_index": str(LABELED_INDEX),
    "visual_root": str(VISUAL_ROOT),
}
with open(PACKAGE_BASE / "phase2_run_manifest.json", "w") as f:
    json.dump(run_manifest, f, indent=2)

if ZIP_PATH.exists():
    ZIP_PATH.unlink()
shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=PACKAGE_BASE)

print("Phase-2 package:", ZIP_PATH)
print(f"Package size: {ZIP_PATH.stat().st_size / 1024**2:.2f} MB")


# visual-baseline stage → Phase 3 hand-off

When the final gate prints:

> **visual-baseline stage CLEARED — READY TO MOVE TO PHASE 3**

visual-baseline stage is frozen around:

- the primary subject-grouped split,
- the 8-second visual temporal context,
- the 27-class semantic action vocabulary, while preserving the frozen 29-label Phase-1 taxonomy and lifecycle audit,
- the canonical dense Temporal Transformer,
- the conventional temporal-mask robustness baseline,
- the fixed clean-data metrics,
- and the fixed 5/15/30/50/70% temporal corruption benchmark.

The next phase may introduce EEG **without changing these visual baselines or their split definitions**. The Phase-1 synchronized index remains the authoritative multimodal join.

## Phase-2 artifact

`/kaggle/working/egobrain_phase2_artifacts.zip`

## Most important machine-readable outputs

- `reports/clean_model_summary.csv`
- `reports/corruption_benchmark.csv`
- `reports/corruption_summary.csv`
- `reports/primary_subject_split.csv`
- `reports/cross_scene_status.json`
- `reports/phase2_gate_report.json`
- `problem_lock.json`
- `models/temporal_transformer/best_weights_only.pt`
- `models/temporal_mask_transformer/best_weights_only.pt`
